In [3]:
import re

import numpy as np
import pandas as pd
from numpy.typing import NDArray
from scipy.stats import wasserstein_distance

from sea_surface_temp import noaa, roi, ssp
from sea_surface_temp.constants import NOAA_OUTPUT_PATH, ROI_INPUT_PATH, SSP_OUTPUT_PATH

In [4]:
SOURCE_IDS = [p.stem for p in (SSP_OUTPUT_PATH / "temperatures").iterdir()]
YEARS = list(range(1982, 2015))

Cargamos la GeoDataFrame con los polígonos de las regiones de interés.

In [5]:
gdf = roi.load_roi_gdf(ROI_INPUT_PATH)
gdf.head(5)

,region_id,geometry
0,HEX0000001,"POLYGON ((-122.0331 28.10519, -122.08865 28.12..."
1,HEX0000002,"POLYGON ((-122.0611 28.2579, -122.11675 28.276..."
2,HEX0000003,"POLYGON ((-122.0893 28.4106, -122.14505 28.429..."
3,HEX0000004,"POLYGON ((-122.11769 28.5633, -122.17354 28.58..."
4,HEX0000005,"POLYGON ((-122.14628 28.71598, -122.20222 28.7..."


Generamos una DataFrame con base en los datos históricos de la NOAA. Cada fila corresponde a una región, y cada columna a un año.

In [6]:
df_noaa = (
    pd.concat(
        [
            noaa.generate_roi_hottest_month_temperature_df(
                NOAA_OUTPUT_PATH, gdf, year=year
            )
            .set_index("region_id")
            .assign(year=year)
            .dropna(subset=["temperature_c"])
            for year in YEARS
        ]
    )
    .reset_index()
    .pivot_table(
        index="region_id",
        columns="year",
        values="temperature_c",
    )
)
df_noaa.head(5)

year,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
region_id,,,,,,,,,,,,,,,,,,,,,
HEX0000001,20.822166,21.613787,20.070386,21.462086,21.037069,20.480133,20.161151,20.201928,21.465663,19.352934,...,20.586219,20.878839,20.601633,19.733701,21.323435,19.370346,19.814106,18.893971,20.546317,21.895808
HEX0000002,20.701939,21.456776,19.787098,21.301336,20.878000,20.316454,20.038666,20.079033,21.541609,19.457333,...,20.401669,20.610645,20.605665,19.588709,21.198997,19.323225,19.768064,18.773663,20.301664,21.608002
HEX0000003,20.693551,21.446462,19.773328,21.296269,20.868599,20.308011,20.028248,20.071028,21.544095,19.461228,...,20.394332,20.598843,20.604042,19.581974,21.195356,19.318676,19.763980,18.771743,20.290471,21.598530
HEX0000004,20.503868,21.213224,19.461935,21.181667,20.656002,20.117094,19.792665,19.889999,21.600321,19.595482,...,20.302580,20.331936,20.567333,19.429678,21.113001,19.245806,19.671614,18.728334,20.037334,21.451611
HEX0000005,20.324811,21.031610,19.231251,21.030040,20.496925,19.939433,19.658509,19.730264,21.497539,19.663848,...,20.167705,20.179622,20.491402,19.322009,21.060614,19.171394,19.596269,18.668020,19.867927,21.443242


Definimos una función para hacer lo mismo pero ahora con los datos de un cierto modelo histórico (especificado por `source_id` and `member_id`).

In [7]:
def results_to_dataframe(*, source_id: str, member_id: str) -> pd.DataFrame:
    return (
        pd.concat(
            [
                ssp.generate_roi_hottest_month_temperature_df(
                    SSP_OUTPUT_PATH,
                    gdf,
                    source_id=source_id,
                    member_id=member_id,
                    year=year,
                )
                .set_index("region_id")
                .assign(year=year)
                .dropna(subset=["temperature_c"])
                for year in YEARS
            ]
        )
        .reset_index()
        .pivot_table(
            index="region_id",
            columns="year",
            values="temperature_c",
        )
    )

Por ejemplo, obtenemos las temperaturas para `source_id=CESM2` y `member_id=r1i1p1f1`: 

In [8]:
results_to_dataframe(source_id="CESM2", member_id="r1i1p1f1").head(5)

year,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
region_id,,,,,,,,,,,,,,,,,,,,,
HEX0000001,21.935677,19.995493,21.594632,21.239470,21.394824,20.322887,21.368925,21.906002,22.097623,21.404232,...,22.034285,23.310124,21.385022,21.747297,21.238611,21.136415,23.340044,22.811109,21.191996,23.172598
HEX0000002,21.922781,19.984043,21.574368,21.214836,21.381851,20.315756,21.358673,21.892105,22.081545,21.398113,...,22.027031,23.308376,21.383064,21.720966,21.235250,21.124537,23.325935,22.813843,21.196888,23.173483
HEX0000003,21.922781,19.984043,21.574368,21.214836,21.381851,20.315756,21.358673,21.892105,22.081545,21.398113,...,22.027031,23.308376,21.383064,21.720966,21.235250,21.124537,23.325935,22.813843,21.196888,23.173483
HEX0000004,21.922781,19.984043,21.574368,21.214836,21.381851,20.315756,21.358673,21.892105,22.081545,21.398113,...,22.027031,23.308376,21.383064,21.720966,21.235250,21.124537,23.325935,22.813843,21.196888,23.173483
HEX0000005,21.922781,19.984043,21.574368,21.214836,21.381851,20.315756,21.358673,21.892105,22.081545,21.398113,...,22.027031,23.308376,21.383064,21.720966,21.235250,21.124537,23.325935,22.813843,21.196888,23.173483


Como recordatorio, cada fuente de datos está definida por dos identificadores:

- `source_id`: Identificador del modelo utilizado.
- `member_id`: Identificador de los parámetros y suposiciones utilizadas. Este es un string con el formato `r[R]i[I]p[P]f[F]`, donde las letras entre paréntesis representant un número. Cada número identifica un aspecto distinto de la simulación de un modelo:

| ID | Nombre | Explicación |
| ---           | ---    |   ---       |
|`I`            | Initialization | Diferentes maneras de construir el estado inicial (usar distintas fuentes de datos, considerar distintos fenómenos, usar distintas observaciones históricas, etc.) |   
| `R`           | Realization | Diferentes *estados iniciales*. Aún depués de determinar la manera de construir el estado inicial, no es posible saber con absoluta certeza las condiciones iniciales; siempre existe una cierta incertidumbre debido a errores en los instrumentos, zonas o períodos no observados, carencia de datos históricos, etc. Una *realización* hace "suposiciones" sobre estas variables desconocidas. La idea es ejecutar muchas realizaciones utilizando distintos supuestos, y posteriormente obtener un intervalo de confianza de los resultados.<br>Por ejemplo, para el modelo CESM1, los investigadores crearon distintas realizaciones variando la temperatura inicial aleatoriamente por $10^{-14}$ grados Celsius. A pesar de esta minúscula variación, las predicciones finales acabaron siendo considerablemente diferentes, debido al efecto mariposa |
| `P` | Physics | Diferentes versiones de los parámetros físicos del modelo y simulación de nubes. |
| `F` | Forcing | Diferentes versiones de datos utilizados para "forzar" el modelo en una dirección específica. Estos datos suelen ser observaciones reales para un cierto período de tiempo. Sirven para guiar al modelo para que no se desvíe mucho de la realidad. | 

Por ejemplo, el experimento `MRI-ESM2-0` tiene los siguientes 6 archivos:
- `r1i1p1f1`
- `r2i1p1f1`
- `r3i1p1f1`
- `r4i1p1f1`
- `r5i1p1f1`
- `r1i2p1f1`

Esto significa que el modelo con parámetros `i=1, p=1, f=1` tuvo **cinco realizaciones**, mientras que el modelo con parámetros `i=2, p=1, f=1` tuvo **una realización**; en otras palabras, para un cierto año y región de interés, tenemos **5 resultados posibles** bajo el primer modelo. 

Obviamente, lo ideal sería agregar estas 5 temperaturas en una sola. Para esto, calculamos la **mediana** para cada región de interés y año, a lo largo de todas las realizaciones. Usamos la mediana en vez de la media para evitar que una predicción extrema no sesgue demasiado el resultado.

In [ ]:
# Aquí vamos a guardar las temperaturas de cada una de las variaciones del
# experimento. Es un diccionario donde las llaves son
# (<nombre del experimento>, <initialization>, <physics>, <forcing>), y los valores
# un arreglo. Cada arreglo contiene la mediana de las temperaturas para todas las
# zonas de interés, obtenida a lo largo de todas las realizaciones. Por ejemplo:
#
# {
#    (CESM2, 1, 1, 1): [27, 32, 25, 21]
# }     |    |  |  |     \____________/
#     Nombre I  P  F         Mediana
ensemble_medians: dict[tuple[str, int, int, int], pd.DataFrame] = {}

# Aquí vamos a guardar el número de realizaciones para cada experimento.
realization_counts: dict[tuple[str, int, int, int], int] = {}

# Iteramos sobre cada experimento
for source_id in SOURCE_IDS:
    # Para obtener la mediana, es necesario tener una lista de las temperaturas de
    # todas las realizaciones. Las guardamos en esta variable. Por ejemplo:
    #
    # {
    #    (CESM2, 1, 1, 1): [[27, 32, 25, 21], [24, 31, 26, 24]]
    # }     |    |  |  |     \____________/    \____________/
    #     Nombre I  P  F      Realización 1     Realización 2
    member_temperatures: dict[tuple[str, int, int, int], list[NDArray[np.float64]]] = {}

    # Iteramos sobre cada member_id
    for member_id_path in (SSP_OUTPUT_PATH / "temperatures" / source_id).iterdir():
        # Obtenemos el nombre del archivo (e.g, r1i1p1f1)
        member_id = member_id_path.stem

        # Lo partimos para obtener los valores de cada parámetro
        # (1, 1, 1, 1 en el ejemplo)
        _, initialization, physics, forcing = (
            int(value)
            for value in re.findall(r"r(\d+)i(\d+)p(\d+)f(\d+)", member_id)[0]
        )

        # Definimos la llave del diccionario
        # (<nombre del experimento>, <initialization>, <physics>, <forcing>)
        configuration = (source_id, initialization, physics, forcing)

        # Cargamos los resultados del experimento correspondiente y lo convertimos
        # a una DataFrame
        temperatures = (
            results_to_dataframe(source_id=source_id, member_id=member_id)
            # Es importante que tenga los mismos índices que la DataFrame histórica
            # de la NOAA para poder comparar ambas
            .reindex(index=df_noaa.index, columns=YEARS)
        )

        # Extraemos el arreglo de temperaturas y lo añadimos al diccionario
        values = temperatures.to_numpy(dtype=float, na_value=np.nan)
        member_temperatures.setdefault(configuration, []).append(
            np.where(np.isfinite(values), values, np.nan)
        )

    # Después de añadir todos los arreglos al diccionario, calculamos la mediana
    # de temperaturas a lo largo de las realizaciones
    for configuration, members in member_temperatures.items():
        ensemble_medians[configuration] = pd.DataFrame(
            np.median(np.stack(members), axis=0),
            index=df_noaa.index,
            columns=YEARS,
        )
        realization_counts[configuration] = len(members)

Por ejemplo, podemos revisar la tabla obtenida para el modelo con 5 realizaciones mencionado previamente:

In [18]:
ensemble_medians[("MRI-ESM2-0", 1, 1, 1)].head(5)

,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
region_id,,,,,,,,,,,,,,,,,,,,,
HEX0000001,22.518653,23.174379,22.796429,23.081986,22.778299,22.441279,23.183211,22.735532,22.634053,22.828523,...,23.095605,22.779494,23.291118,22.685389,23.057593,23.339341,23.276669,23.152156,22.961759,24.119987
HEX0000002,22.514071,23.180176,22.796625,23.082520,22.774349,22.455894,23.186169,22.738728,22.636017,22.835810,...,23.097191,22.773247,23.292862,22.679834,23.060331,23.326717,23.271112,23.152143,22.968315,24.113688
HEX0000003,22.514071,23.180176,22.796625,23.082520,22.774349,22.455894,23.186169,22.738728,22.636017,22.835810,...,23.097191,22.773247,23.292862,22.679834,23.060331,23.326717,23.271112,23.152143,22.968315,24.113688
HEX0000004,22.514071,23.180176,22.796625,23.082520,22.774349,22.455894,23.186169,22.738728,22.636017,22.835810,...,23.097191,22.773247,23.292862,22.679834,23.060331,23.326717,23.271112,23.152143,22.968315,24.113688
HEX0000005,22.514071,23.180176,22.796625,23.082520,22.774349,22.455894,23.186169,22.738728,22.636017,22.835810,...,23.097191,22.773247,23.292862,22.679834,23.060331,23.326717,23.271112,23.152143,22.968315,24.113688


Ahora, tenemos una tabla de temperaturas modeladas para cada modelo y experimento. Aparte, tenemos una tabla de temperaturas históricas de la NOAA. Queremos una manera de comparar cada modelo y obtener una "puntuación". Idealmente, entre mejor sea esta puntuación, más cerca se apegan los resultados modelados a la realidad observada. 

La manera más sencilla de hacer esto es usando el **Mean Absolute Error**. Para una región $r$ este se define como:

$$
\text{MAE}_r = \frac{1}{N}\sum_{y\in Y} |H_r(y) - M_r(y)|
$$

Donde $H_r(y)$ es la temperatura histórica y $M_r(y)$ es la mediana de las temperaturas modeladas entre las realizaciones de una misma configuración, para el año $y$ y la región $r$. $Y$ es el conjunto de todos los años (en nuestro caso de 1982 a 2014).

Incluimos dos métricas adicionales para cubrir diferentes aspectos de la comparación. El MAE seguirá siendo nuestra métrica principal, pero estas adiciones nos permiten capturar otros aspectos del modelado:

- **RMSE:** Lo usamos para dar más peso a los errores grandes y detectar configuraciones con discrepancias importantes en algunos años que pueden quedar menos reflejadas en el MAE.
- **Distancia de Wasserstein:** La usamos como diagnóstico del nivel y la variabilidad de las temperaturas del mes más caliente, sin exigir que los años cálidos y fríos coincidan con los observados. Esto es útil porque la variabilidad interna de las simulaciones históricas puede no estar sincronizada con la observada ([diseño experimental de CMIP6](https://gmd.copernicus.org/articles/9/1937/2016/gmd-9-1937-2016.pdf)), aunque esta distancia no evalúa la tendencia ni el orden temporal.

In [ ]:
# Distancia de Wasserstein
def w1_common_years(
    modelled: NDArray[np.float64], observed: NDArray[np.float64]
) -> float:
    valid = np.isfinite(modelled) & np.isfinite(observed)

    if not valid.any():
        return np.nan

    return float(wasserstein_distance(modelled[valid], observed[valid]))


# Función para calcular todas las puntuaciones para un cierto modelo
def score_results(observed: pd.DataFrame, modelled: pd.DataFrame) -> pd.DataFrame:
    observed_reindexed = observed.reindex(
        index=modelled.index, columns=modelled.columns
    )
    modelled_values = modelled.to_numpy(dtype=float, na_value=np.nan)
    observed_values = observed_reindexed.to_numpy(dtype=float, na_value=np.nan)
    valid = np.isfinite(modelled_values) & np.isfinite(observed_values)
    errors = (modelled - observed_reindexed).where(valid)
    mae = errors.abs().mean(axis=1)
    rmse = np.sqrt(errors.pow(2).mean(axis=1))

    return pd.DataFrame(
        {
            "wasserstein": [
                w1_common_years(model_row, observed_row)
                for model_row, observed_row in zip(
                    modelled_values, observed_values, strict=True
                )
            ],
            "mae": mae,
            "rmse": rmse,
            "n_years": valid.sum(axis=1),
        },
        index=modelled.index,
    )

Calculando la puntuación para el ejemplo anterior:

In [ ]:
score_results(
    observed=df_noaa, modelled=ensemble_medians[("MRI-ESM2-0", 1, 1, 1)]
).head(5)

,wasserstein,mae,rmse,n_years
region_id,,,,
HEX0000001,2.362766,2.362766,2.495302,33
HEX0000002,2.483891,2.483891,2.604371,33
HEX0000003,2.490387,2.490387,2.610204,33
HEX0000004,2.626433,2.626433,2.736100,33
HEX0000005,2.743395,2.743395,2.843801,33


Aplicándolo a los resultados de todos los modelos:

In [22]:
scores = {
    configuration: score_results(observed=df_noaa, modelled=temperatures)
    for configuration, temperatures in ensemble_medians.items()
}

Las tablas producidas tienen una fila por cada región de interés. Para obtener una única puntuación por modelo, promediamos las métricas a lo largo de todas las regiones:

In [36]:
scores[("MRI-ESM2-0", 1, 1, 1)].mean()

wasserstein     1.170814
mae             1.223339
rmse            1.342166
n_years        30.828291
dtype: float64

(Como nota, el hecho de que tengamos una puntuación para cada región puede sernos útil si eventualmente queremos combinar los resultados de varios modelos. Por ejemplo, hay algunos modelos que funcionan mejor en el Pacífico que en el Atlántico, o que incrementan su error entre más crece la latitud.)

Finalmente, promediamos los resultados de todos los modelos para obtener el ranking final:

In [37]:
def rank_results(
    result_map: dict[tuple[str, int, int, int], pd.DataFrame],
    realization_counts: dict[tuple[str, int, int, int], int],
) -> pd.DataFrame:
    distances = pd.concat(
        {config: result["wasserstein"] for config, result in result_map.items()},
        axis=1,
    )
    mae_scores = pd.concat(
        {config: result["mae"] for config, result in result_map.items()},
        axis=1,
    )
    rmse_scores = pd.concat(
        {config: result["rmse"] for config, result in result_map.items()},
        axis=1,
    )
    counts = pd.concat(
        {config: result["n_years"] for config, result in result_map.items()},
        axis=1,
    )
    distances.columns.names = [
        "source_id",
        "I",
        "P",
        "F",
    ]
    counts.columns.names = distances.columns.names
    mae_scores.columns.names = distances.columns.names
    rmse_scores.columns.names = distances.columns.names

    eligible = (np.isfinite(distances) & counts.eq(len(YEARS))).all(axis=1)
    w = distances.loc[eligible]
    mae = mae_scores.loc[eligible]
    rmse = rmse_scores.loc[eligible]

    if w.empty:
        msg = (
            "No ROIs have valid scores for all configured years in every configuration."
        )
        raise ValueError(msg)

    ranking = pd.DataFrame(
        {
            "mean_w1": w.mean(axis=0, skipna=False),
            "median_w1": w.median(axis=0, skipna=False),
            "p90_w1": w.quantile(0.9, axis=0),
            "mean_mae": mae.mean(axis=0, skipna=False),
            "median_mae": mae.median(axis=0, skipna=False),
            "p90_mae": mae.quantile(0.9, axis=0),
            "mean_rmse": rmse.mean(axis=0, skipna=False),
            "median_rmse": rmse.median(axis=0, skipna=False),
            "p90_rmse": rmse.quantile(0.9, axis=0),
            "n_regions": len(w),
        }
    )
    ranking["n_realizations"] = [
        realization_counts[configuration] for configuration in ranking.index
    ]
    return ranking.sort_values("mean_mae")

In [38]:
ranked = rank_results(scores, realization_counts)
ranked

mean_w1  median_w1    p90_w1  mean_mae  median_mae  \
source_id     I P F                                                        
KACE-1-0-G    1 1 1  0.816969   0.746912  1.564128  0.967259    0.865429   
MRI-ESM2-0    1 1 1  1.215813   1.116763  2.712834  1.266938    1.133596   
              2 1 1  1.211745   1.139772  2.673111  1.307529    1.172710   
EC-Earth3-Veg 1 1 1  1.270739   0.976817  2.804384  1.371614    1.076306   
CESM2-WACCM   1 1 1  1.314134   1.223687  2.601688  1.374692    1.252343   
CESM2         1 1 1  1.361413   1.276154  2.713345  1.405510    1.286272   
GFDL-ESM4     1 1 1  1.373086   0.991399  3.521348  1.437350    1.001477   
EC-Earth3     1 1 1  1.420342   1.111007  2.831749  1.464928    1.178741   

                      p90_mae  mean_rmse  median_rmse  p90_rmse  n_regions  \
source_id     I P F                                                          
KACE-1-0-G    1 1 1  1.582512   1.096876     0.992827  1.694801      34804   
MRI-ESM2-0    1 1 1  2.716425   1.385673     1.221663  2.906226      34804   
              2 1 1  2.678288   1.455491     1.278703  2.943280      34804   
EC-Earth3-Veg 1 1 1  2.809335   1.558390     1.239863  3.105405      34804   
CESM2-WACCM   1 1 1  2.611463   1.521832     1.374225  2.886828      34804   
CESM2         1 1 1  2.717540   1.523754     1.378228  2.945489      34804   
GFDL-ESM4     1 1 1  3.521684   1.569047     1.119337  3.705281      34804   
EC-Earth3     1 1 1  2.832317   1.582703     1.270166  2.983351      34804   

                     n_realizations  
source_id     I P F                  
KACE-1-0-G    1 1 1               3  
MRI-ESM2-0    1 1 1               5  
              2 1 1               1  
EC-Earth3-Veg 1 1 1               1  
CESM2-WACCM   1 1 1               3  
CESM2         1 1 1              11  
GFDL-ESM4     1 1 1               3  
EC-Earth3     1 1 1              15

Esto nos dice que, para nuestras regiones de interés, el modelo que mejor desempeño histórico tiene es `KACE-1-0-G`. 